# TensorFlow device fallback: CUDA → Intel XPU → CPU (WSL-friendly)

This notebook does two things:

1. **Implements a simple runtime fallback** for TensorFlow compute:
   - Try **CUDA GPU** (`/GPU:0`)
   - If that fails, try **Intel XPU** (`/XPU:0`, via ITEX)
   - If that fails, fall back to **CPU** (`/CPU:0`)

2. Includes a **detailed setup guide** for the known-good stack:
   **WSL Ubuntu 24.04 + Intel Arc 140V + Python 3.11 + TensorFlow 2.15.1 + ITEX 2.15.0.3**.

> Notes
> - Seeing CUDA warnings in logs is normal if you don't have NVIDIA drivers in WSL.
> - For Intel XPU, you need `intel_extension_for_tensorflow` installed and Level Zero visible (e.g., `sycl-ls` shows `[level_zero:gpu]`).


In [1]:
import os, sys, platform
import tensorflow as tf

print("Python:", sys.version)
print("Platform:", platform.platform())
print("TensorFlow:", tf.__version__)
print("CUDA GPUs:", tf.config.list_physical_devices("GPU"))
print("CPUs:", tf.config.list_physical_devices("CPU"))


2026-03-04 12:22:20.223874: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-04 12:22:20.239728: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-03-04 12:22:20.270898: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-03-04 12:22:20.270925: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-03-04 12:22:20.271958: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to

Python: 3.11.14 (main, Oct 10 2025, 08:54:04) [GCC 13.3.0]
Platform: Linux-6.6.87.2-microsoft-standard-WSL2-x86_64-with-glibc2.39
TensorFlow: 2.15.1
CUDA GPUs: []
CPUs: [PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')]


In [2]:
# Try to import Intel Extension for TensorFlow (ITEX).
# If it's not installed, XPU fallback will be skipped.

itex = None
try:
    import intel_extension_for_tensorflow as itex
    print("ITEX:", itex.__version__)
except Exception as e:
    print("ITEX not available:", repr(e))

print("XPU devices:", tf.config.list_physical_devices("XPU"))


ITEX: 2.15.0.3
XPU devices: [PhysicalDevice(name='/physical_device:XPU:0', device_type='XPU')]


## Optional: XPU memory controls (growth + cap)

These must run **before** you create tensors/models to have any effect. Some backends may not support them; we catch exceptions.

In [3]:
# --- XPU memory growth (best-effort) ---
xpus = tf.config.list_physical_devices("XPU")
for d in xpus:
    try:
        tf.config.experimental.set_memory_growth(d, True)
        print("Enabled XPU memory growth on:", d)
    except Exception as e:
        print("XPU memory growth not supported on:", d, "->", e)

# --- XPU memory cap (best-effort) ---
# NOTE: If you already created tensors/models in this kernel session, restart the kernel before trying this.
XPU_MEMORY_LIMIT_MB = 4096

if xpus:
    try:
        tf.config.set_logical_device_configuration(
            xpus[0],
            [tf.config.LogicalDeviceConfiguration(memory_limit=XPU_MEMORY_LIMIT_MB)]
        )
        print(f"Capped XPU[0] memory to {XPU_MEMORY_LIMIT_MB} MB")
        print("Logical XPUs:", tf.config.list_logical_devices("XPU"))
    except Exception as e:
        print("Could not cap XPU memory (backend may not support it):", e)


Enabled XPU memory growth on: PhysicalDevice(name='/physical_device:XPU:0', device_type='XPU')
Could not cap XPU memory (backend may not support it): Virtual devices are not supported for XPU


## Fallback runner: CUDA → XPU → CPU

We force a small matmul onto each device in order. If the device isn't present or initialization fails, we move to the next.

In [4]:
import traceback
import time

def matmul_smoke_test(device: str, n: int = 2048):
    """Run a simple matmul on a specific TF device string and return (ok, info)."""
    t0 = time.time()
    with tf.device(device):
        a = tf.random.normal([n, n])
        b = tf.random.normal([n, n])
        c = tf.matmul(a, b)
        # Force execution and fetch a tiny value
        v = c[0, 0].numpy()
        dev = c.device
    dt = time.time() - t0
    return True, {"device": dev, "seconds": dt, "value00": float(v)}

def try_device(device: str):
    try:
        ok, info = matmul_smoke_test(device)
        print(f"✅ SUCCESS on {device}")
        print("   Tensor device string:", info["device"])
        print("   Elapsed (s):", round(info["seconds"], 3))
        return True, info
    except Exception as e:
        print(f"❌ FAILED on {device}: {type(e).__name__}: {e}")
        # Uncomment next line if you want the full traceback:
        # traceback.print_exc()
        return False, None

def pick_best_device():
    # 1) CUDA GPU (TensorFlow uses 'GPU' for CUDA devices)
    gpus = tf.config.list_physical_devices("GPU")
    if gpus:
        if try_device("/GPU:0")[0]:
            return "/GPU:0"

    # 2) Intel XPU (ITEX)
    xpus = tf.config.list_physical_devices("XPU")
    if xpus:
        if try_device("/XPU:0")[0]:
            return "/XPU:0"

    # 3) CPU
    if try_device("/CPU:0")[0]:
        return "/CPU:0"

    return None

chosen = pick_best_device()
print("\nChosen device:", chosen)


2026-03-04 12:22:55.641568: I tensorflow/core/common_runtime/next_pluggable_device/next_pluggable_device_factory.cc:118] Created 1 TensorFlow NextPluggableDevices. Physical device type: XPU


✅ SUCCESS on /XPU:0
   Tensor device string: /job:localhost/replica:0/task:0/device:XPU:0
   Elapsed (s): 10.778

Chosen device: /XPU:0


## Explicit Intel XPU test (requested snippet)

This forces a matmul onto `/XPU:0` (will error if XPU is unavailable).

In [5]:
print("XPU:", tf.config.list_physical_devices("XPU"))

with tf.device("/XPU:0"):
    a = tf.random.normal([2048, 2048])
    b = tf.random.normal([2048, 2048])
    c = tf.matmul(a, b)
    tf.print("matmul ok, shape:", tf.shape(c), "device:", c.device)


XPU: [PhysicalDevice(name='/physical_device:XPU:0', device_type='XPU')]
matmul ok, shape: [2048 2048] device: /job:localhost/replica:0/task:0/device:XPU:0


# Setup guide (known-good): WSL Ubuntu 24.04 + Arc 140V + Python 3.11 + TF 2.15.1 + ITEX 2.15.0.3

This section is a practical end‑to‑end guide for reproducing the working stack.

## 1) Windows host prerequisites (driver + WSL plumbing)

1. Update WSL (PowerShell as Admin):
   ```powershell
   wsl --update
   wsl --shutdown
   ```
2. Install the **latest Intel Graphics Windows driver** for your Arc/Lunar Lake GPU. Reboot after installing.

WSL uses a Windows-side driver for GPU access, and Linux-in-WSL provides user-space runtimes.

## 2) WSL checks (must pass)

Inside WSL:
```bash
ls -l /dev/dxg
uname -r
```

- If `/dev/dxg` exists: GPU is exposed to WSL.

## 3) Install/verify Intel GPU runtimes in WSL (Level Zero + OpenCL)

Install core runtime packages (many systems already have these when OpenCL/Level Zero works):

```bash
sudo apt update
sudo apt install -y ocl-icd-libopencl1 clinfo intel-opencl-icd libze1
```

Verify OpenCL sees the Intel GPU:
```bash
clinfo | grep -E "Platform Name|Device Name|Device Type" -n | head -n 140
```

## 4) Install oneAPI Base Toolkit (optional but very useful)

If installed, enable it each shell:
```bash
source /opt/intel/oneapi/setvars.sh
```

Check Level Zero / SYCL device visibility:
```bash
sycl-ls
```

You want to see a line like:
`[level_zero:gpu] ... Intel(R) Graphics ...`

## 5) Python venv + pinned packages (the known-good combo)

Create venv:
```bash
python3.11 -m venv .venv_itex
source .venv_itex/bin/activate
python -m pip install -U pip wheel setuptools
```

Install the exact working versions:
```bash
pip install tensorflow==2.15.1
pip install intel-extension-for-tensorflow==2.15.0.3
pip install intel-extension-for-tensorflow-lib==2.15.0.3.2
```

Pin them in `requirements.txt` to avoid accidental breakage.

## 6) Validate TensorFlow sees XPU

```bash
python - <<'PY'
import tensorflow as tf
import intel_extension_for_tensorflow as itex
print("TF:", tf.__version__)
print("ITEX:", itex.__version__)
print("XPU devices:", tf.config.list_physical_devices("XPU"))
PY
```

Expected: `XPU devices: [PhysicalDevice(name='/physical_device:XPU:0', ...)]`

## 7) Smoke test on XPU

```python
import tensorflow as tf
import intel_extension_for_tensorflow as itex

with tf.device("/XPU:0"):
    a = tf.random.normal([2048, 2048])
    b = tf.random.normal([2048, 2048])
    c = tf.matmul(a, b)
    tf.print("matmul ok, shape:", tf.shape(c), "device:", c.device)
```

## 8) Memory controls (optional)

- **Memory growth** (best-effort): `tf.config.experimental.set_memory_growth(...)`
- **Memory cap** (best-effort): `tf.config.set_logical_device_configuration(... memory_limit=...)`

Both must run **before** TensorFlow initializes the device (i.e., before tensors/models are created).
